# Worst-Case Decision Importance

`toy_examples.ipynb` asks whether DECIDE recovers the right importances when surrogate models do not. This notebook asks a different question: whether the *choice of* $\Lambda$ and $\Phi$ changes what DECIDE itself can see.

The framework leaves the summary function $\Lambda$ and the spread function $\Phi$ free precisely so that the analyst can pick a risk profile. With $\Lambda$ = mean and $\Phi$ = variance we recover Sobol' indices and measure a decision's *average* influence. With $\Lambda$ = max and $\Phi$ = range we measure its *worst-case* influence instead. The claim to be tested is that these are not cosmetic variations: there are multiverses in which the average-case reading is not merely less precise but actively misleading, and the decision that can do the most damage is the one the average-case reading ranks last.

The toy below is that multiverse. It is constructed so that the ranking is exactly inverted between the two instantiations.

In [ ]:
# imports
import itertools
import numpy as np
import pandas as pd
import sys
sys.path.append('../..')
from src.metrics.metrics import organize_multiverse_results, decide

## The construction

Twelve binary judgment calls, 4096 universes, in three groups.

- **Five routine decisions** `r1`-`r5`, each with a plain main effect between 0.044 and 0.060 accuracy. These behave the way an analyst expects: each one shifts the outcome by a modest amount, in every background context.
- **Six trigger decisions** `t1`-`t6`, each *mildly beneficial* on its own, worth +0.02 accuracy. There is nothing to warn an analyst away from any of them, and each looks like a small improvement in isolation.
- **One catastrophic decision** `CAT`, which does nothing at all -- unless every one of the six trigger decisions is set to 1, in which case accuracy collapses by 0.30.

$$\text{accuracy} = 0.80 + \sum_{i=1}^{5} \beta_i r_i + 0.02 \sum_{j=1}^{6} t_j - 0.30 \cdot \text{CAT} \cdot \prod_{j=1}^{6} t_j$$

The catastrophe fires in 32 of the 4096 universes, or 0.78% of the multiverse, and costs 0.30 accuracy wherever it fires -- enough to take the best-performing corner of the multiverse to below its worst-performing one.

Note that every decision here is binary. That is load-bearing rather than incidental, for reasons discussed at the end.

In [ ]:
# 12 binary judgment calls, 2^12 = 4096 universes.
#
#   r1..r5  routine decisions, each with a plain main effect
#   t1..t6  trigger decisions, each mildly beneficial on its own (+0.02)
#   CAT     harmless on its own, but costs 30 accuracy points when every
#           trigger decision is set to 1
ROUTINE = [0.060, 0.056, 0.052, 0.048, 0.044]
TRIGGER_EFFECT = 0.02
N_TRIGGER = 6
CATASTROPHE = 0.30
BASE = 0.60

decisions = ([f'r{i + 1}' for i in range(len(ROUTINE))]
             + [f't{i + 1}' for i in range(N_TRIGGER)]
             + ['CAT'])
X = np.array(list(itertools.product([0, 1], repeat=len(decisions))))

n_routine = len(ROUTINE)
triggers_all_on = X[:, n_routine:n_routine + N_TRIGGER].all(axis=1)
catastrophe_fires = triggers_all_on & (X[:, -1] == 1)

accuracy = (BASE
            + X[:, :n_routine] @ np.array(ROUTINE)
            + TRIGGER_EFFECT * X[:, n_routine:n_routine + N_TRIGGER].sum(axis=1)
            - CATASTROPHE * catastrophe_fires)

mv_toy = pd.DataFrame(X, columns=decisions).assign(accuracy=accuracy)

print("universes:", len(mv_toy))
print("accuracy range:", round(accuracy.min(), 3), "to", round(accuracy.max(), 3))
print("catastrophe fires in", int(catastrophe_fires.sum()), "universes",
      f"({100 * catastrophe_fires.mean():.2f}%)")

## Reading the multiverse three ways

The first two rows below are the framework's two natural instantiations. The third is an intermediate that keeps $\Phi$ = range but replaces $\Lambda$ = max with an exceedance frequency: the proportion of comparable sets in which the decision can swing accuracy by more than 0.15. It reports how *often* a decision can do serious damage rather than how much damage it can do at its worst.

In [ ]:
# the same multiverse read through three (Lambda, Phi) instantiations
mv = organize_multiverse_results(mv_toy, decisions, ['accuracy'])

INSTANTIATIONS = [
    ('mean/var', np.var, np.mean),
    ('max/range', np.ptp, np.max),
    ('P(swing>0.15)', np.ptp, lambda a: float(np.mean(np.asarray(a) > 0.15))),
]

scores = {}
for label, spread, summary in INSTANTIATIONS:
    result = decide(mv, 'accuracy', max_order=1, spread=spread, summary=summary)
    scores[label] = {d[0]: v['total_effect'] for d, v in result.items()}

table = pd.DataFrame(scores)
table['role'] = (['routine'] * n_routine + ['trigger'] * N_TRIGGER + ['CATASTROPHIC'])
table['rank mean/var'] = table['mean/var'].rank(ascending=False, method='min').astype(int)
table['rank max/range'] = table['max/range'].rank(ascending=False, method='min').astype(int)
table

In [ ]:
# CAT is last on average and first in the tail
print("CAT rank under mean/var :", table.loc['CAT', 'rank mean/var'], "of", len(decisions))
print("CAT rank under max/range:", table.loc['CAT', 'rank max/range'], "of", len(decisions))
print()
print("max/range, CAT vs the best routine decision:",
      round(table.loc['CAT', 'max/range'] / table['max/range'][:n_routine].max(), 2), "x")
print("mean/var, CAT vs the worst routine decision:",
      round(table.loc['CAT', 'mean/var'] / table['mean/var'][:n_routine].min(), 2), "x")

## What the existing methods report

In [ ]:
# what the existing surrogate-based methods report on the same multiverse
from sklearn.linear_model import LinearRegression
from src.metrics.functional_anova import MultiverseFanova

lm = LinearRegression().fit(mv_toy[decisions].to_numpy(), accuracy)

# NOTE: MultiverseFanova does not pass a seed to fANOVA, so the forest is
# randomised on every call and these numbers move slightly between runs. The
# ranking of CAT does not.
fanova_scores = (MultiverseFanova(features=mv_toy[decisions], outcome=pd.Series(accuracy))
                 .quantify_individual_importance()['individual importance'])

comparison = pd.DataFrame({
    'role': table['role'],
    'lm |coef|': pd.Series(np.abs(lm.coef_), index=decisions),
    'fANOVA': fanova_scores,
    'DECIDE mean/var': table['mean/var'],
    'DECIDE max/range': table['max/range'],
}).loc[decisions]
for col in ['lm |coef|', 'fANOVA', 'DECIDE mean/var', 'DECIDE max/range']:
    comparison[f'rk {col.split()[-1]}'] = comparison[col].rank(ascending=False, method='min').astype(int)
comparison

In [ ]:
# what DECIDE points at: the comparable set where CAT does its damage
others = [d for d in decisions if d != 'CAT']
grouped = mv_toy.groupby(others, sort=False)['accuracy']
swings = grouped.max() - grouped.min()

print("CAT's comparable sets:", len(swings))
print("  sets where CAT changes nothing:", int((swings == 0).sum()))
print("  sets where CAT costs 0.30     :", int((swings > 0.29).sum()))
print()

worst_context = dict(zip(others, swings.idxmax()))
rows = mv_toy
for name, value in worst_context.items():
    rows = rows[rows[name] == value]
print("worst comparable set for CAT -- every trigger is set to 1:")
print({k: int(v) for k, v in worst_context.items()})
print(rows[['CAT', 'accuracy']].to_string(index=False))